# **Lab 4: Naïve Bayes, Text Classification, and Sentiment Analysis**

## Objectives
In this lab, you will learn about:

- Implementing Naïve Bayes classifiers for text classification
- Handling text preprocessing and feature extraction
- Building sentiment analysis models
- Evaluating classification performance
- Working with English and Arabic texts


# Exercise 1: Basic Text Classification with Multinomial Naïve Bayes

## Objective
Implement a basic Multinomial Naïve Bayes classifier for binary text classification on movie reviews.

---

### Tasks:

#### Task 1: Load and Preprocess the Dataset
1. **Dataset**: Use the IMDB movie reviews dataset, which contains movie reviews labeled as positive or negative.
2. **Exploration Steps**:
   - Load the dataset and print the total number of instances.
   - Print the number of instances in each class (positive and negative).
   - Display a few sample reviews from each class to understand the data.

3. **Data Preprocessing**:
   - Split the reviews into training and test sets (80% and 20%).
   - Create a vocabulary from the training data.
   - Convert the texts into a **bag-of-words** (BoW) representation by counting occurrences of each word in the vocabulary.
   

#### Task 2: Implement the Multinomial Naïve Bayes Classifier from Scratch

The **Multinomial Naïve Bayes** classifier uses Bayes' theorem to classify a document by calculating the probability of each class given the words in the document.

---

### Explanation and Equations

1. **Bayes’ Theorem for Classification**

   For a given review $ \mathbf{x} $, we assign it to the class $ c $ that maximizes the posterior probability $ P(c \mid \mathbf{x}) $:

   $$
   \hat{y} = \arg \max_{c \in C} P(c \mid \mathbf{x})
   $$

   Using Bayes' theorem, we get:

   $$
   P(c \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid c) \cdot P(c)}{P(\mathbf{x})}
   $$

   Since $ P(\mathbf{x}) $ is constant across classes, we can simplify to:

   $$
   P(c \mid \mathbf{x}) \propto P(\mathbf{x} \mid c) \cdot P(c)
   $$

2. **Calculate Prior Probabilities $ P(c) $**

   The prior probability $ P(c) $ is the probability of each class $ c $ in the training set:

   $$
   P(c) = \frac{\text{number of documents in class } c}{\text{total number of documents}}
   $$

3. **Calculate Conditional Probabilities $ P(\text{word} \mid c) $**

   - The conditional probability $ P(\text{word} \mid c) $ represents the likelihood of each word appearing in a document given the class.

   - For a word $ w_i $ in a document, we calculate $ P(w_i \mid c) $ as follows, using **Laplace smoothing** to handle zero probabilities:

     $$
     P(w_i \mid c) = \frac{\text{count of } w_i \text{ in documents of class } c + \alpha}{\text{total words in class } c + \alpha \cdot V}
     $$

     where:
     - $ \alpha $ is a smoothing parameter (commonly set to 1 for Laplace smoothing).
     - $ V $ is the total number of unique words in the vocabulary.

4. **Prediction Using Log Probabilities**

   For a new review $ \mathbf{x} $, the classifier computes the log probability of each class to prevent underflow:

   $$
   \log P(c \mid \mathbf{x}) = \log P(c) + \sum_{i=1}^{n} x_i \cdot \log P(w_i \mid c)
   $$

   - Here, $ x_i $ represents the count of word $ w_i $ in the review $ \mathbf{x} $.
   - The class with the highest log probability is assigned as the predicted class.

---

#### Task 3: Evaluate the Classifier

1. **Evaluation Metrics**:
   - Calculate accuracy, precision, recall, and F1 score on the test set.
2. **Confusion Matrix**:
   - Generate a confusion matrix to visualize classification results and highlight areas of improvement.
3. **Comparison**:
   - Compare the results with and without stopword removal to understand its impact on classification performance.
   - Compare the results with and without the usage of either stemming or lemmatization to understand their impact on classification performance.


In [2]:
import nltk
nltk.download('punkt')    # For tokenization
nltk.download('stopwords') # For stopwords removal
nltk.download('wordnet')  # For lemmatization

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Rosey\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rosey\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rosey\AppData\Roaming\nltk_data...


True

In [5]:
"""
IMDB Movie Review Sentiment Analysis using Naive Bayes
"""

import re
import csv
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import math

def preprocess_text(text, stopwords_dict, remove_stopwords=True, stem=False, lemmatize=False):
    """
    Purpose:
        Transforms raw text into a cleaned list of tokens suitable for machine learning.

    Algorithm Steps:
        1. Remove HTML tags using regex pattern
        2. Convert text to lowercase
        3. Extract only words using regex pattern (word boundaries)
        4. Apply optional filtering and normalization

    Arguments:
        text (str): Raw text input (e.g., "This movie was GREAT! <br />")
        stopwords_dict (set): Set of stopwords to remove (e.g., {"this", "was"})
        remove_stopwords (bool): Whether to filter out common words
        stem (bool): Whether to reduce words to their root form (e.g., "running" -> "run")
        lemmatize (bool): Whether to reduce words to their dictionary form

    Returns:
        list: Processed words (e.g., ["movie", "great"])

    Example:
        >>> text = "This movie was GREAT! <br /><br /> I really enjoyed it."
        >>> stopwords_dict = {"this", "was", "i", "really", "it"}
        >>> preprocess_text(text, stopwords_dict, remove_stopwords=True)
        ["movie", "great", "enjoyed"]
    """
    # Remove HTML tags
    # Your code here
    text = re.sub(r'<.*?>', ' ', text)
    # Convert to lowercase
    text = text.lower()
    # Your code here

    # Tokenize words using regex
    words = re.findall(r'\b\w+\b', text)

    # Remove stopwords
    if remove_stopwords:
        words = [word for word in words if word not in stopwords_dict]
    # Apply stemming or lemmatization if needed
    if stem:
        ps = PorterStemmer()
        words = [ps.stem(word) for word in words]
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        words = [lemmatizer.lemmatize(word) for word in words]
   # Remove stopwords
    if remove_stopwords:
        words = [word for word in words if word not in stopwords_dict]
    return words  # list of tokens

def load_imdb_data(file_path):
    """
    Purpose:
        Loads and preprocesses the IMDB dataset from a CSV file, splitting it into training and test sets.

    File Format Requirements:
        CSV file with two columns:
        1. 'review': Text of the movie review
        2. 'sentiment': 'positive' or 'negative'

    Algorithm Steps:
        1. Initialize empty lists for reviews and labels
        2. Load stopwords for preprocessing
        3. Read CSV file line by line:
           - Preprocess each review
           - Convert sentiment labels to binary format
        4. Split data into training (80%) and test (20%) sets

    Arguments:
        file_path (str): Path to the IMDB dataset CSV file

    Returns:
        tuple: (X_train, X_test, y_train, y_test)
        - X_train, X_test: Lists of preprocessed review tokens
        - y_train, y_test: Lists of corresponding sentiment labels

    Example:
        >>> X_train, X_test, y_train, y_test = load_imdb_data("IMDB Dataset.csv")
        >>> print(X_train[0][:5])  # First 5 words of first review
        ["wonderful", "film", "great", "acting", "plot"]
        >>> print(y_train[0])
        "pos"
    """
    reviews = []
    labels = []

    # Load stopwords
    stopwords_dict = set(stopwords.words('english'))

    # Read CSV file
    with open(file_path, 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            # Process review text
            processed_review = preprocess_text(row['review'], stopwords_dict)
            reviews.append(processed_review)

            # Convert sentiment labels
            label = 'pos' if row['sentiment'] == 'positive' else 'neg'
            labels.append(label)

    # Split into train (80%) and test (20%) sets
    return train_test_split(reviews, labels, test_size=0.2, random_state=42)

class SimpleNaiveBayes:
    """
    Multinomial Naive Bayes classifier for text classification.

    Mathematical Foundation:
        Naive Bayes uses Bayes' Theorem: P(c|d) = P(c) * P(d|c) / P(d)
        where:
        - P(c|d) is probability of class c given document d
        - P(c) is prior probability of class c
        - P(d|c) is probability of document d given class c
        - P(d) is probability of document d

        For text classification:
        1. P(c) = number of documents in class c / total documents
        2. P(w|c) = (count of word w in class c + 1) / (total words in class c + |V|)
           where |V| is vocabulary size (Laplace smoothing)
        3. P(d|c) = product of P(w|c) for each word w in document d

    Log Probability:
        We use log probabilities to prevent underflow:
        log(P(c|d)) = log(P(c)) + sum(log(P(w|c))) for each word w in d
    """

    def __init__(self):
        """
        Initialize the classifier with empty data structures:
        - class_priors: Dictionary storing P(c) for each class
        - word_probs: Dictionary storing P(w|c) for each word and class
        - vocabulary: Set of all unique words in training data
        """
        self.class_priors = {}  # Stores P(class)
        self.word_probs = {}    # Stores P(word|class)
        self.vocabulary = set()  # All unique words

    def fit(self, documents, labels):
        """
        Purpose:
            Train the Naive Bayes classifier on the given documents.

        Mathematical Steps:
            1. Calculate class priors P(c):
               P(c) = count(documents in class c) / total documents

            2. Build vocabulary:
               Create set of all unique words across all documents

            3. Calculate word probabilities P(w|c) with Laplace smoothing:
               P(w|c) = (count(w in class c) + 1) / (total words in c + |V|)
               where |V| is vocabulary size

        Arguments:
            documents (list): List of preprocessed documents [[word1, word2, ...], ...]
            labels (list): List of corresponding labels ["pos", "neg", ...]

        Example:
            >>> docs = [["great", "movie"], ["terrible", "film"]]
            >>> labels = ["pos", "neg"]
            >>> classifier.fit(docs, labels)
            >>> print(classifier.class_priors)
            {"pos": 0.5, "neg": 0.5}
        """
        # Step 1: Count documents per class and calculate priors
        total_docs = len(documents)
        class_counts = {}
         # Count how many documents in each class
        for label in labels:
            class_counts[label] = class_counts.get(label, 0) + 1
    
        # Calculate P(c) = count(class c) / total documents
        for class_label, count in class_counts.items():
            self.class_priors[class_label] = count / total_docs
            
        # Step 2: Create vocabulary of all unique words
        for document in documents:
            self.vocabulary.update(document)
            
        vocab_size = len(self.vocabulary)
        word_counts_per_class = {}  # {class: {word: count}}
        total_words_per_class = {}  # {class: total_word_count}
        # Step 3 : calculate word probabilities with laplace smoothing
        for class_label in class_counts.keys():
            word_counts_per_class[class_label] = {}
            total_words_per_class[class_label] = 0
        for doc, label in zip(documents, labels):
            for word in doc:
                # Count this word for this class
                if word not in word_counts_per_class[label]:
                    word_counts_per_class[label][word] = 0
                word_counts_per_class[label][word] += 1
                total_words_per_class[label] += 1

         # P(w|c) = (count(w in class c) + 1) / (total words in c + |V|)
        for class_label  in class_counts.keys():
            self.word_probs[class_label]={}
            for word in self.vocabulary:
                word_count = word_counts_per_class[class_label].get(word, 0)
                self.word_probs[class_label][word] = ((word_count+1)/(total_words_per_class[class_label]+vocab_size))
             
        


    def predict(self, documents):
        """
        Purpose:
            Predict class labels for given documents using the trained model.

        Mathematical Steps:
            For each document and class, calculate:
            log(P(c|d)) = log(P(c)) + sum(log(P(w|c)))

            1. Start with log of class prior: log(P(c))
            2. Add log probabilities of each word: sum(log(P(w|c)))
            3. Select class with highest log probability

        Arguments:
            documents (list): List of preprocessed documents to classify

        Returns:
            list: Predicted labels for each document

        Example:
            >>> docs = [["amazing", "movie"], ["terrible", "waste"]]
            >>> classifier.predict(docs)
            ["pos", "neg"]
        """
        predictions = [] # Store predicted labels
        for doc in documents:
            class_scores = {}
            for class_label in self.class_priors.keys():
                # Start with log prior
                log_prob = math.log(self.class_priors[class_label])
                # Add log probabilities of each word
                for word in doc:
                    if word in self.vocabulary:
                        log_prob += math.log(self.word_probs[class_label].get(word, 1 / (sum(self.word_probs[class_label].values()) + len(self.vocabulary))))
                class_scores[class_label] = log_prob
            # Select class with highest score
            predicted_class = max(class_scores, key=class_scores.get)
            predictions.append(predicted_class)

        return predictions

def analyze_reviews(documents, labels):
    """
    Purpose:
        Calculate statistical metrics about review lengths for each sentiment class.

    Statistical Metrics:
        - Minimum length
        - Maximum length
        - Average length
        - Total number of reviews

    Arguments:
        documents (list): List of preprocessed documents
        labels (list): List of corresponding sentiment labels

    Returns:
        dict: Dictionary containing statistics for each class

    Example:
        >>> docs = [["great", "movie"], ["terrible", "waste", "time"]]
        >>> labels = ["pos", "neg"]
        >>> analyze_reviews(docs, labels)
        {
            "pos": {"min": 2, "max": 2, "avg": 2.0, "total_reviews": 1},
            "neg": {"min": 3, "max": 3, "avg": 3.0, "total_reviews": 1}
        }
    """
    stats = {}
    for label in set(labels):
        lengths = [len(doc) for doc, doc_label in zip(documents, labels) if doc_label == label]
        stats[label] = {
            "min": min(lengths),
            "max": max(lengths),
            "avg": sum(lengths) / len(lengths),
            "total_reviews": len(lengths)
        }
    return stats

def get_top_words(classifier, n=10):
    """
    Purpose:
        Identify the most indicative words for each sentiment class based on their probabilities.

    Algorithm Steps:
        1. For each class (positive/negative):
           - Sort words by their conditional probability P(w|c)
           - Select top n words with highest probabilities

    Arguments:
        classifier (SimpleNaiveBayes): Trained classifier instance
        n (int): Number of top words to return for each class

    Returns:
        dict: Dictionary with top words and their probabilities for each class

    Example:
        >>> top_words = get_top_words(classifier, n=2)
        >>> print(top_words)
        {
            "pos": [("great", 0.1), ("amazing", 0.08)],
            "neg": [("terrible", 0.1), ("boring", 0.07)]
        }
    """
    top_words = {}

    for class_label in classifier.class_priors.keys():  
        # Sort words by their probabilities for this class
        sorted_words = sorted(classifier.word_probs[class_label].items(), key=lambda item: item[1], reverse=True)
        # Select top n words
        top_words[class_label] = sorted_words[:n]

    return top_words

def print_detailed_analysis(classifier, X_test, y_test, stats, top_words):
    """
    Purpose:
        Generate and display a comprehensive analysis of the classifier's performance
        and dataset characteristics.

    Output Metrics:
        1. Use Classification Report from sklearn.metrics
           - Precision: True positives / (True positives + False positives)
           - Recall: True positives / (True positives + False negatives)
           - F1-score: 2 * (Precision * Recall) / (Precision + Recall)
        2. Dataset Statistics:
           - Review counts and length distributions by sentiment
        3. Most indicative words by sentiment

    Arguments:
        classifier (SimpleNaiveBayes): Trained classifier
        X_test (list): Test set reviews
        y_test (list): Test set labels
        stats (dict): Review statistics
        top_words (dict): Most indicative words by sentiment
    """
    # Predict on test set
    y_pred = classifier.predict(X_test) 
    # Print classification report
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=["neg", "pos"]))
    # Print dataset statistics
    print("Dataset Statistics:")
    for sentiment, sentiment_stats in stats.items():
        print(f"Sentiment: {sentiment}")
        print(f"  Total Reviews: {sentiment_stats['total_reviews']}")
        print(f"  Min Length: {sentiment_stats['min']}")
        print(f"  Max Length: {sentiment_stats['max']}")
        print(f"  Avg Length: {sentiment_stats['avg']:.2f}")
    # Print most indicative words
    print("Most Indicative Words by Sentiment:")
    for sentiment, words in top_words.items():
        print(f"Sentiment: {sentiment}")
        for word, prob in words:
            print(f"  Word: {word}, P(word|{sentiment}): {prob:.6f}")


# Main execution
if __name__ == "__main__":
    # Load and preprocess IMDB dataset
    print("Loading and preprocessing IMDB dataset...")
    X_train, X_test, y_train, y_test = load_imdb_data("IMDB Dataset.csv")

    # Train classifier
    print("Training Naive Bayes classifier...")
    classifier = SimpleNaiveBayes()
    classifier.fit(X_train, y_train)

    # Analyze reviews
    stats = analyze_reviews(X_train, y_train)

    # Get top words
    top_words = get_top_words(classifier, n=20)

    # Print detailed analysis
    print_detailed_analysis(classifier, X_test, y_test, stats, top_words)

Loading and preprocessing IMDB dataset...
Training Naive Bayes classifier...
Classification Report:
              precision    recall  f1-score   support

         neg       0.85      0.88      0.86      4961
         pos       0.87      0.84      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000

Dataset Statistics:
Sentiment: pos
  Total Reviews: 19961
  Min Length: 8
  Max Length: 1435
  Avg Length: 121.35
Sentiment: neg
  Total Reviews: 20039
  Min Length: 4
  Max Length: 824
  Avg Length: 117.64
Most Indicative Words by Sentiment:
Sentiment: neg
  Word: movie, P(word|neg): 0.016277
  Word: film, P(word|neg): 0.012296
  Word: one, P(word|neg): 0.008636
  Word: like, P(word|neg): 0.007357
  Word: even, P(word|neg): 0.004975
  Word: bad, P(word|neg): 0.004813
  Word: good, P(word|neg): 0.004812
  Word: would, P(word|neg): 0.004590
  Word: time, P(word|neg): 

# Exercise 2: Sentiment Analysis with Binary Naïve Bayes

## Objective
Implement Binary Naïve Bayes for sentiment analysis and compare the results to those obtained in exercise 1.

---

### Tasks:

#### Task 1: Load and Explore the Dataset
Use the same dataset as in exercise 1.

---

### Task 2: Implement Multinomial Naïve Bayes (NB) and Binary Naïve Bayes Classifiers

#### Explanation and Equations

**Binary Naïve Bayes Classifier**

The **Binary Naïve Bayes classifier** uses the presence or absence of a word, rather than its frequency, for prediction.

- **Binary Likelihood** $ P(w_i \mid c) $:

  Instead of counting occurrences, Binary NB only considers whether each word is present in the instance (1 if present, 0 if absent):

  $$
  P(w_i \mid c) = \frac{\text{number of reviews in class } c \text{ containing } w_i + \alpha}{\text{total reviews in class } c + 2 \alpha}
  $$

  Here, $\alpha$ is a smoothing parameter that helps to handle the cases where a word might not appear in a given class. This prevents zero probabilities for unseen words, thus ensuring that all words have a chance to contribute to the predictions.

- **Prediction**: For a review, predict the class using the same maximization rule but with presence/absence values:

  $$
  \hat{y} = \arg \max_{c \in C} \left( \log P(c) + \sum_{i=1}^{n} \text{presence}(w_i) \cdot \log P(w_i \mid c) \right)
  $$

---

### Task 3: Compare Performance of Both Classifiers

1. **Evaluation Metrics**:
   - Calculate accuracy, precision, recall, and F1 score for both classifiers on the test set.

2. **Analyze Word Contributions**:
   - For each model, identify which words are most strongly associated with positive or negative sentiment by examining the conditional probabilities $ P(w_i \mid c) $.

3. **Binary NB vs. Multinomial NB**:
   - Identify cases where Binary NB performs better or worse than Multinomial NB and explain potential reasons based on word presence vs. frequency effects.


In [7]:
"""
IMDB Movie Review Sentiment Analysis using Binary Naive Bayes
"""

import re
import csv
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import math
from collections import defaultdict

def preprocess_text(text, stopwords_dict, lemmatize=False):
    """
    Purpose:
        Transforms raw text into a set of unique tokens for binary representation.

    Algorithm Steps:
        1. Remove HTML tags
        2. Convert to lowercase
        3. Extract words using regex
        4. Remove stopwords
        5. Lemmatize words (optional)
        6. Convert to set for binary representation

    Arguments:
        text (str): Raw review text
        stopwords_dict (set): Set of stopwords to remove
        lemmatize (bool): Whether to apply lemmatization

    Returns:
        set: Unique processed words

    Example:
        >>> text = "This movie was great! The movie was amazing! <br />"
        >>> stopwords_dict = {"this", "was", "the"}
        >>> preprocess_text(text, stopwords_dict)
        {"movie", "great", "amazing"}  # Duplicates removed automatically
    """
    # Remove HTML tags  
    text = re.sub(r'<.*?>', ' ', text)
    # Convert to lowercase
    text = text.lower()
    # Tokenize words using regex
    words = re.findall(r'\b\w+\b', text)
    # Remove stopwords
    words = [word for word in words if word not in stopwords_dict]
    # Apply lemmatization if needed
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        words = [lemmatizer.lemmatize(word) for word in words]
    return set(words)  # Return unique words as a set


def load_imdb_data_binary(file_path):
    """
    Purpose:
        Loads IMDB dataset and converts reviews to binary word presence representation.

    File Format Requirements:
        Same as Exercise 1: CSV with 'review' and 'sentiment' columns

    Algorithm Steps:
        1. Load stopwords
        2. Process each review to get unique words (binary presence)
        3. Split into train/test sets

    Arguments:
        file_path (str): Path to IMDB dataset CSV

    Returns:
        tuple: (X_train, X_test, y_train, y_test)
        - X_train, X_test: Lists of sets containing unique words
        - y_train, y_test: Lists of sentiment labels

    Example:
        >>> X_train, X_test, y_train, y_test = load_imdb_data_binary("IMDB Dataset.csv")
        >>> print(X_train[0])  # First review's unique words
        {"wonderful", "film", "great", "acting", "plot"}
    """
    reviews = []
    labels = []

    stopwords_dict = set(stopwords.words('english'))

    with open(file_path, 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            # Get unique words (binary presence)
            processed_review = preprocess_text(row['review'], stopwords_dict)
            reviews.append(processed_review)

            label = 'pos' if row['sentiment'] == 'positive' else 'neg'
            labels.append(label)

    return train_test_split(reviews, labels, test_size=0.2, random_state=42)

class BinaryNaiveBayes:
    """
    Binary Naive Bayes classifier for text classification.

    Mathematical Foundation:
        Uses Bayes' Theorem with binary word presence:
        P(c|d) ∝ P(c) * ∏ P(w|c) for w in d

        Key differences from Multinomial NB:
        1. P(w|c) = (docs containing w in class c + α) / (docs in class c + 2α)
           where α is smoothing parameter (typically 1)
        2. Only considers word presence (1 or 0), not frequency

    Log Probability:
        log(P(c|d)) = log(P(c)) + ∑ log(P(w|c)) for w in d
    """

    def __init__(self, alpha=1.0):
        """
        Initialize classifier with smoothing parameter alpha.

        Data structures:
        - class_priors: P(c) for each class
        - word_presence: Count of documents containing each word by class
        - class_docs: Number of documents in each class
        - vocabulary: All unique words
        """
        self.alpha = alpha
        self.class_priors = {}
        self.word_presence = defaultdict(lambda: defaultdict(int))
        self.class_docs = defaultdict(int)
        self.vocabulary = set()

    def fit(self, documents, labels):
        """
        Purpose:
            Train Binary Naive Bayes on document sets.

        Mathematical Steps:
            1. Calculate P(c) = docs in class / total docs
            2. For each word w and class c:
               P(w|c) = (docs with w in c + α) / (docs in c + 2α)

        Arguments:
            documents (list): List of sets containing unique words
            labels (list): Corresponding sentiment labels

        Example:
            >>> docs = [{"great", "movie"}, {"terrible", "waste"}]
            >>> labels = ["pos", "neg"]
            >>> classifier.fit(docs, labels)
        """
        total_docs = len(documents)

        # Count documents per class
        for label in labels:
            self.class_docs[label] += 1 
        # Calculate class priors P(c)   
        for class_label, count in self.class_docs.items():
            self.class_priors[class_label] = count / total_docs
        # Build vocabulary and count word presence
        for doc, label in zip(documents, labels):
            for word in doc:
                self.vocabulary.add(word)
                self.word_presence[label][word] += 1
        # Calculate word probabilities P(w|c) with Laplace smoothing
        vocab_size = len(self.vocabulary)
        for class_label in self.class_docs.keys():
            total_class_docs = self.class_docs[class_label]
            for word in self.vocabulary:
                word_count = self.word_presence[class_label].get(word, 0)
                self.word_presence[class_label][word] = (word_count + self.alpha) / (total_class_docs + 2 * self.alpha)
    def predict(self, documents):
        """
        Purpose:
            Predict class labels using binary word presence.

        Mathematical Steps:
            For each doc and class, calculate:
            log(P(c|d)) = log(P(c)) + ∑ log(P(w|c))
            where w are present words only

        Arguments:
            documents (list): List of sets containing unique words

        Returns:
            list: Predicted labels

        Example:
            >>> docs = [{"amazing", "film"}, {"boring", "waste"}]
            >>> classifier.predict(docs)
            ["pos", "neg"]
        """
        predictions = []
        for doc in documents:
            class_scores = {}
            for class_label in self.class_priors.keys():
                log_prob = math.log(self.class_priors[class_label])
                for word in doc:
                    if word in self.vocabulary:
                        log_prob += math.log(self.word_presence[class_label].get(word, self.alpha / (self.class_docs[class_label] + 2 * self.alpha)))
                class_scores[class_label] = log_prob
            predicted_class = max(class_scores, key=class_scores.get)
            predictions.append(predicted_class)

        return predictions

def get_distinctive_words(classifier, n=10):
    """
    Purpose:
        Find words most strongly associated with each sentiment.

    Algorithm Steps:
        1. For each word, calculate presence ratio:
           ratio = P(w|pos) / P(w|neg)
        2. Sort words by ratio for each class

    Arguments:
        classifier (BinaryNaiveBayes): Trained classifier
        n (int): Number of top words to return

    Returns:
        dict: Top distinctive words and their ratios by class

    Example:
        >>> words = get_distinctive_words(classifier, n=2)
        >>> print(words)
        {
            "pos": [("excellent", 5.2), ("amazing", 4.8)],
            "neg": [("terrible", 4.1), ("boring", 3.9)]
        }
    """
    top_words = {}
    for class_label in classifier.class_priors.keys():  
        # Sort words by their probabilities for this class
        sorted_words = sorted(classifier.word_presence[class_label].items(), key=lambda item: item[1], reverse=True)
        # Select top n words
        top_words[class_label] = sorted_words[:n]

    return top_words

def compare_classifiers(binary_pred, multi_pred, y_test):
    """
    Purpose:
        Compare performance of Binary and Multinomial NB.

    Output:
        1. Classification reports for both models
        2. Analysis of disagreements

    Arguments:
        binary_pred (list): Binary NB predictions
        multi_pred (list): Multinomial NB predictions
        y_test (list): True labels
    """
    print("\n=== Model Comparison ===")

    print("\nBinary Naive Bayes Performance:")
    print(classification_report(y_test, binary_pred))

    print("\nMultinomial Naive Bayes Performance:")
    print(classification_report(y_test, multi_pred))

    # Analyze disagreements
    disagreements = sum(1 for b, m in zip(binary_pred, multi_pred) if b != m)
    print(f"\nDisagreements between models: {disagreements}")
    print(f"Disagreement rate: {disagreements/len(y_test)*100:.4f}%")

# Main execution
if __name__ == "__main__":
    # Load and preprocess IMDB dataset
    print("Loading and preprocessing IMDB dataset...")
    X_train, X_test, y_train, y_test = load_imdb_data_binary("IMDB Dataset.csv")

    # Train Binary Naive Bayes
    print("Training Binary Naive Bayes classifier...")
    binary_classifier = BinaryNaiveBayes()
    binary_classifier.fit(X_train, y_train)

    # Make predictions
    binary_predictions = binary_classifier.predict(X_test)

    # Get distinctive words
    distinctive_words = get_distinctive_words(binary_classifier, n=20)

 # Print results
    print("\n=== Binary Naive Bayes Results ===")
    print("\nClassification Report:")
    print(classification_report(y_test, binary_predictions, target_names=['neg','pos']))


Loading and preprocessing IMDB dataset...
Training Binary Naive Bayes classifier...

=== Binary Naive Bayes Results ===

Classification Report:
              precision    recall  f1-score   support

         neg       0.87      0.84      0.86      4961
         pos       0.85      0.88      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000



# Exercise 3: Handling Negation in Sentiment Analysis

## Objective
Improve sentiment classification from Exercise 2 by implementing negation handling.

---

### Tasks:

#### Task 1: Implement Negation Marking
1. **Negation Handling**: Enhance the text preprocessing pipeline to handle negation, as negation can flip the sentiment of specific words.
   - **Negation Terms**: Define a list of common negation terms such as *not*, *no*, *never*, and *cannot*.
   - **Negation Marking**: When a negation term is encountered, mark all following words with a **`NOT_`** prefix until reaching a punctuation mark (e.g., period, comma, exclamation mark).
2. **Custom Test Cases**:
   - Create custom sentences that include negation, both with and without punctuation boundaries.
   - Test the function by verifying if the `NOT_` prefix is applied correctly.

---

### Negation Marking Approach

1. **Identify Negation Terms and Boundaries**:
   - Start by defining negation terms that typically reverse sentiment, like *not*, *no*, *never*, and *cannot*.
   - Treat punctuation marks as boundaries to stop negation influence since they usually signify the end of a sentence or clause.

2. **Apply Negation Prefix**:
   - For each word in a sentence:
     - If a negation term is encountered, begin prefixing the following words with `NOT_` until a punctuation boundary is reached.
   - Example:
     - Original sentence: "I do not like this movie."
     - With negation marking: "I do NOT_like NOT_this NOT_movie."
   
   - This technique allows the model to learn that *NOT_like* might be associated with negative sentiment, distinguishing it from *like*, which would have a positive connotation without negation.

3. **Custom Test Cases**:
   - Test the function with sentences that include negations with and without punctuation.
   - Example custom test cases:
     - "I do not think it is great." should become "I do NOT_think NOT_it NOT_is NOT_great."
     - "This movie is not amazing, but it’s okay." should become "This movie is NOT_amazing, but it’s okay."

---

#### Task 2: Compare Classification Results

1. **Without Negation Handling**:
   - First, apply the sentiment classifier from Exercise 2 without using negation handling to establish a baseline performance.

2. **With Negation Handling**:
   - Implement the modified classifier that includes negation handling during preprocessing.
   - Train and test the classifier again, this time with negation-marked data.

3. **Performance Comparison**:
   - Evaluate both classifiers on accuracy, precision, recall, and F1 score.
   - Analyze differences between the two versions, noting any improvement in sentiment detection.

4. **Error Analysis**:
   - Identify examples that were misclassified without negation handling but corrected with it.
   - Observe cases where negation handling failed or had no impact, and consider potential improvements.


In [8]:
"""
IMDB Movie Review Sentiment Analysis with Negation Handling

"""

import re
import csv
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import math
from collections import defaultdict

class NegationHandler:
    """
    Purpose:
        Handle negation in text by marking words following negation terms.

    Approach:
        1. Identify negation terms (e.g., "not", "never", "no")
        2. Mark words following negation until punctuation boundary
        3. Special handling for contractions (e.g., "don't", "isn't")

    Example:
        Input: "This movie is not good, but the acting is amazing"
        Output: "This movie is NOT_good, but the acting is amazing"
    """

    def __init__(self):
        """Initialize negation terms and punctuation boundaries"""
        self.negation_terms = {
            'not', 'no', 'never', 'nothing', 'nowhere', 'noone', 'none', 'not',
            "n't", 'cant', 'cannot', 'wont', 'wouldnt', 'shouldnt', 'doesnt',
            'isnt', 'aint', 'hadnt', 'hasnt', 'wasnt', 'werent', 'havent'
        }

        self.punctuation = {'.', ',', '!', '?', ';', ':', '(', ')', '[', ']'}
        self.NOT_PREFIX = 'NOT_'

    def handle_negation(self, text):
        """
        Purpose:
            Process text and mark words affected by negation.

        Algorithm Steps:
            1. Tokenize text
            2. Identify negation terms
            3. Mark subsequent words until punctuation
            4. Reconstruct text with negation marking

        Arguments:
            text (str): Input text to process

        Returns:
            str: Processed text with negation marking

        Example:
            >>> handler = NegationHandler()
            >>> text = "I do not like this movie at all."
            >>> handler.handle_negation(text)
            "I do NOT_like NOT_this NOT_movie NOT_at NOT_all."
        """
        tokens = word_tokenize(text)
        negation_active = False
        processed_tokens = []   
        for token in tokens:
            lower_token = token.lower()
            if lower_token in self.negation_terms:
                negation_active = True
                processed_tokens.append(token)
            elif token in self.punctuation:
                negation_active = False
                processed_tokens.append(token)
            else:
                if negation_active:
                    processed_tokens.append(self.NOT_PREFIX + token)
                else:
                    processed_tokens.append(token)
        return ' '.join(processed_tokens)


def preprocess_with_negation(text, stopwords_dict, negation_handler):
    """
    Purpose:
        Preprocess text with negation handling and return binary word presence.

    Algorithm Steps:
        1. Handle negation marking
        2. Remove HTML tags
        3. Extract words (including negated forms)
        4. Remove stopwords (except negation terms)
        5. Convert to set for binary representation

    Arguments:
        text (str): Raw text input
        stopwords_dict (set): Stopwords to remove
        negation_handler (NegationHandler): Instance for negation processing

    Returns:
        set: Unique processed words including negated forms

    Example:
        >>> text = "This movie is not good, but the acting is great!"
        >>> preprocess_with_negation(text, stopwords, handler)
        {"movie", "NOT_good", "acting", "great"}
    """
    # Step 1: Handle negation
    negated_text = negation_handler.handle_negation(text)
    # Step 2: Remove HTML tags
    negated_text = re.sub(r'<.*?>', ' ', negated_text)
    # Step 3: Tokenize words using regex
    words = re.findall(r'\b\w+\b', negated_text.lower())
    # Step 4: Remove stopwords (except negation terms)
    processed_words = [word for word in words if word not in stopwords_dict or word.startswith(negation_handler.NOT_PREFIX)]
    return set(processed_words)  # Return unique words as a set 


class NegationAwareBinaryNB:
    """
    Binary Naive Bayes classifier with negation awareness.

    Mathematical Foundation:
        Uses modified Bayes' Theorem accounting for negated terms:
        P(c|d) ∝ P(c) * ∏ P(w|c) for w in d

        Where w can be either regular or negated words:
        - P(w|c) for regular words
        - P(NOT_w|c) for negated words

    Key difference:
        - Vocabulary includes both regular and negated forms
        - Separate probability estimates for regular and negated terms
    """

    def __init__(self, alpha=1.0):
        """Initialize classifier with smoothing parameter"""
        self.alpha = alpha
        self.class_priors = {}
        self.word_presence = defaultdict(lambda: defaultdict(int))
        self.class_docs = defaultdict(int)
        self.vocabulary = set()

    def fit(self, documents, labels):
        """
        Purpose:
            Train classifier on documents with negation handling.

        Mathematical Steps:
            1. P(c) = docs in class / total docs
            2. For each word w (regular or negated) and class c:
               P(w|c) = (docs with w in c + α) / (docs in c + 2α)

        Arguments:
            documents (list): List of sets containing words (including negated)
            labels (list): Corresponding sentiment labels

        Example:
            >>> docs = [{"great", "NOT_boring"}, {"terrible", "NOT_good"}]
            >>> labels = ["pos", "neg"]
            >>> classifier.fit(docs, labels)
        """
        total_docs = len(documents)

        # Count documents per class
        for label in labels:
            self.class_docs[label] += 1
        # Calculate class priors P(c)
        for class_label, count in self.class_docs.items():
            self.class_priors[class_label] = count / total_docs
        # Build vocabulary and count word presence
        for doc, label in zip(documents, labels):
            for word in doc:
                self.vocabulary.add(word)
                self.word_presence[label][word] += 1
        # Calculate word probabilities P(w|c) with Laplace smoothing
        vocab_size = len(self.vocabulary)
        for class_label in self.class_docs.keys():
            total_class_docs = self.class_docs[class_label]
            for word in self.vocabulary:
                word_count = self.word_presence[class_label].get(word, 0)
                self.word_presence[class_label][word] = (word_count + self.alpha) / (total_class_docs + 2 * self.alpha)

    def predict(self, documents):
        """
        Purpose:
            Predict sentiment using both regular and negated words.

        Mathematical Steps:
            For each doc and class, calculate:
            log(P(c|d)) = log(P(c)) + ∑ log(P(w|c))
            where w includes both regular and negated words

        Arguments:
            documents (list): List of sets containing words

        Returns:
            list: Predicted labels
        """
        predictions = []
        for doc in documents:
            class_scores = {}
            for class_label in self.class_priors.keys():
                log_prob = math.log(self.class_priors[class_label])
                for word in doc:
                    if word in self.vocabulary:
                        log_prob += math.log(self.word_presence[class_label].get(word, self.alpha / (self.class_docs[class_label] + 2 * self.alpha)))
                class_scores[class_label] = log_prob
            predicted_class = max(class_scores, key=class_scores.get)
            predictions.append(predicted_class)

        return predictions

def analyze_negation_impact(classifier, test_docs, test_labels):
    """
    Purpose:
        Analyze the impact of negation handling on classification.

    Analysis Steps:
        1. Identify most common negated terms
        2. Compare predictions with/without negation
        3. Find examples where negation changed the outcome

    Arguments:
        classifier (NegationAwareBinaryNB): Trained classifier
        test_docs (list): Test documents
        test_labels (list): True labels

    Returns:
        dict: Analysis results

    Example:
        >>> results = analyze_negation_impact(classifier, test_docs, test_labels)
        >>> print(results['common_negated'])
        ["NOT_good", "NOT_like", "NOT_bad"]
    """
    negated_word_counts = defaultdict(int)
    for doc in test_docs:
        for word in doc:
            if word.startswith('NOT_'):
                negated_word_counts[word] += 1

    common_negated = sorted(negated_word_counts.items(), key=lambda item: item[1], reverse=True)[:10]

    return {
        'common_negated': [word for word, count in common_negated]
    }

def test_negation_handler():
    """
    Purpose:
        Test negation handling with custom test cases.

    Test Cases:
        1. Simple negation
        2. Multiple negations
        3. Negation with punctuation
        4. Contractions

    Example:
        >>> test_negation_handler()
        Test Case 1: "I do not like this movie"
        Result: "i do NOT_like NOT_this NOT_movie"
        ...
    """
    handler = NegationHandler()
    test_cases = [
        "I do not like this movie",
        "This isn't good and I don't like it",
        "The movie is not good, but the acting is great",
        "Never have I seen such a boring film",
        "This movie isn't bad, it's actually quite good"
    ]

    print("\nNegation Handling Test Cases:")
    for i, test in enumerate(test_cases, 1):
        result = handler.handle_negation(test)
        print(f"\nTest {i}:")
        print(f"Input:  {test}")
        print(f"Output: {result}")

# Main execution
if __name__ == "__main__":
    # Initialize negation handler
    negation_handler = NegationHandler()

    # Test negation handling
    test_negation_handler()

    # Load and preprocess IMDB dataset with negation
    print("\nLoading and preprocessing IMDB dataset with negation handling...")
    stopwords_dict = set(stopwords.words('english'))

    reviews = []
    labels = []

    with open("IMDB Dataset.csv", 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            processed_review = preprocess_with_negation(
                row['review'],
                stopwords_dict,
                negation_handler
            )
            reviews.append(processed_review)

            label = 'pos' if row['sentiment'] == 'positive' else 'neg'
            labels.append(label)

    X_train, X_test, y_train, y_test = train_test_split(
        reviews, labels, test_size=0.2, random_state=42
    )

    # Train classifier
    print("Training Negation-Aware Binary Naive Bayes classifier...")
    classifier = NegationAwareBinaryNB()
    classifier.fit(X_train, y_train)

    # Make predictions
    predictions = classifier.predict(X_test)

    # Analyze results
    print("\n=== Classification Results with Negation Handling ===")
    print("\nClassification Report:")
    print(classification_report(y_test, predictions))


Negation Handling Test Cases:

Test 1:
Input:  I do not like this movie
Output: I do not NOT_like NOT_this NOT_movie

Test 2:
Input:  This isn't good and I don't like it
Output: This is n't NOT_good NOT_and NOT_I NOT_do n't NOT_like NOT_it

Test 3:
Input:  The movie is not good, but the acting is great
Output: The movie is not NOT_good , but the acting is great

Test 4:
Input:  Never have I seen such a boring film
Output: Never NOT_have NOT_I NOT_seen NOT_such NOT_a NOT_boring NOT_film

Test 5:
Input:  This movie isn't bad, it's actually quite good
Output: This movie is n't NOT_bad , it 's actually quite good

Loading and preprocessing IMDB dataset with negation handling...
Training Negation-Aware Binary Naive Bayes classifier...

=== Classification Results with Negation Handling ===

Classification Report:
              precision    recall  f1-score   support

         neg       0.82      0.89      0.86      4961
         pos       0.89      0.81      0.85      5039

    accuracy    

# Exercise 4: Extend Sentiment Classification with Lexicons

## Objective
Enhance the sentiment classification implemented in Exercise 1 by incorporating features from pre-built sentiment lexicons and evaluate the results.

---

### Tasks:

#### Task 1: Incorporate Lexicon Features
1. **Load a Sentiment Lexicon**:
   - Use the lexicon file provided with this notebook which contain pre-defined lists of positive and negative words commonly associated with sentiment.
   
2. **Create Lexicon Features**:
   - For each text sample, calculate feature counts based on words that match entries in the lexicon.
   - Features to calculate:
     - **Positive Count**: Count the number of positive words in the text (words found in the positive category of the lexicon).
     - **Negative Count**: Count the number of negative words in the text (words found in the negative category of the lexicon).
   - These counts will act as additional features for the classifier, representing overall positive and negative sentiment expressed in the text.

3. **Combine Lexicon Features with Existing Features**:
   - Use the **bag-of-words** (BoW) representation developed in Exercise 1 for word-based features.
   - Combine the BoW features with the lexicon-based features (positive and negative counts) to build a comprehensive feature set.

---

#### Task 2: Evaluate the Impact of Lexicon Features
1. **Baseline Evaluation**:
   - Train and evaluate the original Multinomial Naïve Bayes classifier using only the BoW features from Exercise 1.
   - Record the accuracy, precision, recall, and F1 score.

2. **Lexicon Features Evaluation**:
   - Train and evaluate a new classifier using only the lexicon features (positive and negative counts).
   - Assess the performance to understand the effectiveness of the lexicon as a sentiment analysis tool.

3. **Combined Features Evaluation**:
   - Train and evaluate a classifier using the combined features from the BoW and the lexicon counts.
   - Analyze whether the integration of lexicon features improves sentiment classification performance.

4. **Performance Comparison**:
   - Calculate accuracy, precision, recall, and F1 score for each approach (BoW only, lexicon only, and combined).
   - Discuss the results to see if lexicon features enhance classification performance, particularly in scenarios where word-based features alone may fall short.

---

### Summary and Analysis

- **Analysis of Results**:
  - Observe the impact of lexicon features on the performance of the sentiment classification model.
  - Identify instances where lexicon features improve or degrade classification accuracy compared to using only word-based features.
  - Reflect on why combining features might yield better sentiment classification results by integrating sentiment-specific knowledge from the lexicons.


In [2]:
"""
Exercise 4: Sentiment Classification with Lexicons
Fixed version with all required imports and functions
"""

import re
import csv
import math
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ===== Functions from Exercise 1 =====

def preprocess_text(text, stopwords_dict, remove_stopwords=True, stem=False, lemmatize=False):
    """Preprocess text for classification"""
    text = re.sub(r'<.*?>', ' ', text)
    text = text.lower()
    words = re.findall(r'\b\w+\b', text)
    
    if remove_stopwords:
        words = [word for word in words if word not in stopwords_dict]
    
    if stem:
        ps = PorterStemmer()
        words = [ps.stem(word) for word in words]
    
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        words = [lemmatizer.lemmatize(word) for word in words]
    
    if remove_stopwords:
        words = [word for word in words if word not in stopwords_dict]
    
    return words

def load_imdb_data(file_path):
    """Load and preprocess IMDB dataset"""
    reviews = []
    labels = []
    stopwords_dict = set(stopwords.words('english'))
    
    with open(file_path, 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            processed_review = preprocess_text(row['review'], stopwords_dict)
            reviews.append(processed_review)
            label = 'pos' if row['sentiment'] == 'positive' else 'neg'
            labels.append(label)
    
    return train_test_split(reviews, labels, test_size=0.2, random_state=42)

class SimpleNaiveBayes:
    """Multinomial Naive Bayes classifier"""
    
    def __init__(self):
        self.class_priors = {}
        self.word_probs = {}
        self.vocabulary = set()
    
    def fit(self, documents, labels):
        """Train the classifier"""
        total_docs = len(documents)
        class_counts = {}
        
        # Count documents per class
        for label in labels:
            class_counts[label] = class_counts.get(label, 0) + 1
        
        # Calculate priors
        for class_label, count in class_counts.items():
            self.class_priors[class_label] = count / total_docs
        
        # Build vocabulary
        for document in documents:
            if isinstance(document, list):
                self.vocabulary.update(document)
            elif isinstance(document, dict):
                self.vocabulary.update(document.keys())
        
        vocab_size = len(self.vocabulary)
        word_counts_per_class = {}
        total_words_per_class = {}
        
        # Initialize counters
        for class_label in class_counts.keys():
            word_counts_per_class[class_label] = {}
            total_words_per_class[class_label] = 0
        
        # Count words
        for doc, label in zip(documents, labels):
            if isinstance(doc, list):
                for word in doc:
                    if word not in word_counts_per_class[label]:
                        word_counts_per_class[label][word] = 0
                    word_counts_per_class[label][word] += 1
                    total_words_per_class[label] += 1
            elif isinstance(doc, dict):
                for word, count in doc.items():
                    if word not in word_counts_per_class[label]:
                        word_counts_per_class[label][word] = 0
                    word_counts_per_class[label][word] += count
                    total_words_per_class[label] += count
        
        # Calculate probabilities
        for class_label in class_counts.keys():
            self.word_probs[class_label] = {}
            for word in self.vocabulary:
                word_count = word_counts_per_class[class_label].get(word, 0)
                self.word_probs[class_label][word] = (
                    (word_count + 1) / (total_words_per_class[class_label] + vocab_size)
                )
    
    def predict(self, documents):
        """Predict labels for documents"""
        predictions = []
        
        for doc in documents:
            class_scores = {}
            
            for class_label in self.class_priors.keys():
                log_prob = math.log(self.class_priors[class_label])
                
                if isinstance(doc, list):
                    for word in doc:
                        if word in self.vocabulary:
                            log_prob += math.log(self.word_probs[class_label].get(
                                word, 1 / (sum(self.word_probs[class_label].values()) + len(self.vocabulary))
                            ))
                elif isinstance(doc, dict):
                    for word, count in doc.items():
                        if word in self.vocabulary or word in ['positive_ratio', 'negative_ratio']:
                            if word in self.word_probs[class_label]:
                                log_prob += count * math.log(self.word_probs[class_label][word])
                
                class_scores[class_label] = log_prob
            
            predicted_class = max(class_scores, key=class_scores.get)
            predictions.append(predicted_class)
        
        return predictions

# ===== Exercise 4 Functions =====

def load_lexicon(file_path):
    """
    Load a sentiment lexicon from a file.
    
    Arguments:
        file_path (str): Path to lexicon CSV file
    
    Returns:
        dict: Dictionary with 'positive' and 'negative' word sets
    """
    lexicon = {'positive': set(), 'negative': set()}
    
    with open(file_path, 'r', encoding='utf-8') as file:
        reader = csv.reader(file)
        for row in reader:
            if len(row) >= 2:
                word, sentiment = row[0].strip().lower(), row[1].strip().lower()
                if sentiment == 'positive':
                    lexicon['positive'].add(word)
                elif sentiment == 'negative':
                    lexicon['negative'].add(word)
    
    return lexicon

def create_lexicon_features(text, lexicon):
    """
    Calculate sentiment ratios based on lexicon.
    
    Arguments:
        text (str): Review text
        lexicon (dict): Sentiment lexicon
    
    Returns:
        dict: Positive and negative word ratios
    """
    words = word_tokenize(text.lower())
    total_words = len(words)
    
    if total_words == 0:
        return {'positive_ratio': 0.0, 'negative_ratio': 0.0}
    
    positive_count = sum(1 for word in words if word in lexicon['positive'])
    negative_count = sum(1 for word in words if word in lexicon['negative'])
    
    return {
        'positive_ratio': positive_count / total_words,
        'negative_ratio': negative_count / total_words
    }

def combine_features(bow_features, lexicon_features):
    """
    Combine BoW and lexicon features.
    
    Arguments:
        bow_features (list): List of words (BoW)
        lexicon_features (dict): Lexicon-based features
    
    Returns:
        dict: Combined feature dictionary
    """
    # Convert list to Counter (word frequencies)
    if isinstance(bow_features, list):
        combined = dict(Counter(bow_features))
    else:
        combined = bow_features.copy()
    
    # Add lexicon features
    combined.update(lexicon_features)
    
    return combined

def evaluate_classifier(X_train, y_train, X_test, y_test, feature_type="BoW"):
    """
    Train and evaluate classifier on specified features.
    
    Arguments:
        X_train, X_test: Training and test features
        y_train, y_test: Training and test labels
        feature_type (str): Feature type description
    """
    classifier = SimpleNaiveBayes()
    classifier.fit(X_train, y_train)
    predictions = classifier.predict(X_test)
    
    print(f"\n{'='*60}")
    print(f"Evaluation for {feature_type} Features")
    print('='*60)
    print(classification_report(y_test, predictions, target_names=['neg', 'pos']))

# ===== Main Execution =====

if __name__ == "__main__":
    print("Loading IMDB dataset...")
    X_train, X_test, y_train, y_test = load_imdb_data("IMDB Dataset.csv")
    
    lexicon = load_lexicon("sentiment_lexicon.csv")
    print(f"Lexicon loaded: {len(lexicon['positive'])} positive, {len(lexicon['negative'])} negative words")
    
    X_train_lexicon = [create_lexicon_features(" ".join(doc), lexicon) for doc in X_train]
    X_test_lexicon = [create_lexicon_features(" ".join(doc), lexicon) for doc in X_test]
    
    evaluate_classifier(X_train, y_train, X_test, y_test, feature_type="BoW")
    
    evaluate_classifier(X_train_lexicon, y_train, X_test_lexicon, y_test, feature_type="Lexicon")
    
    X_train_combined = [combine_features(bow, lex) for bow, lex in zip(X_train, X_train_lexicon)]
    X_test_combined = [combine_features(bow, lex) for bow, lex in zip(X_test, X_test_lexicon)]
    
    evaluate_classifier(X_train_combined, y_train, X_test_combined, y_test, feature_type="Combined")
    

Loading IMDB dataset...
Lexicon loaded: 2304 positive, 4153 negative words

Evaluation for BoW Features
              precision    recall  f1-score   support

         neg       0.85      0.88      0.86      4961
         pos       0.87      0.84      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000


Evaluation for Lexicon Features
              precision    recall  f1-score   support

         neg       0.67      0.78      0.72      4961
         pos       0.74      0.63      0.68      5039

    accuracy                           0.70     10000
   macro avg       0.71      0.70      0.70     10000
weighted avg       0.71      0.70      0.70     10000


Evaluation for Combined Features
              precision    recall  f1-score   support

         neg       0.85      0.88      0.86      4961
         pos       0.87      0.84      0.86      5039

    accurac